# Step 4 — Train the RailOps Maintenance Model

Trains a **Random Forest** classifier that flags track segments likely to fail, logs the run to **Vayu MLflow**, and saves `model.joblib` for [Step 5](../05_model_registry/).

- **Features (in this order):** `vibration_hz`, `acoustic_emission_db`, `rail_strain_mu`, `track_temperature_c`, `alignment_deviation_mm`, `last_maintenance_days`
- **Target:** `1` if `failure_probability > 0.5`, else `0`

The dashboard (`07_build_app/app.py`) sends features to Model Serving in exactly this order, so keep `FEATURES` in sync if you change it.

> **Note:** `failure_probability` in the synthetic dataset is computed from these same sensor readings, so accuracy here will be very high and does not reflect real-world performance.

## 1. Configuration

In [ ]:
import os
from pathlib import Path

import joblib
import mlflow
import mlflow.sklearn
import pandas as pd
from dotenv import load_dotenv
from mlflow.models.signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

REPO_ROOT = Path("..").resolve()
load_dotenv(REPO_ROOT / ".env", override=True)

DATASET_PATH = REPO_ROOT / "01_dataset" / "railway_telemetry.csv"
MODEL_PATH = REPO_ROOT / "04_starter-kit" / "model.joblib"
EXPERIMENT_NAME = "railops_predictive_maintenance"
REGISTERED_MODEL_NAME = "RailOps_Maintenance_Model"

FEATURES = [
    "vibration_hz",
    "acoustic_emission_db",
    "rail_strain_mu",
    "track_temperature_c",
    "alignment_deviation_mm",
    "last_maintenance_days",
]
TARGET_THRESHOLD = 0.5

# MLflow reads MLFLOW_TRACKING_USERNAME / MLFLOW_TRACKING_PASSWORD from the environment.
required = ["MLFLOW_TRACKING_URL", "MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD",
            "AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "S3_ENDPOINT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise ValueError(f"Missing in .env: {missing}")

os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ["S3_ENDPOINT"]
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URL"])
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow: {mlflow.get_tracking_uri()} | experiment: {EXPERIMENT_NAME}")

## 2. Load data

In [ ]:
df = pd.read_csv(DATASET_PATH)

X = df[FEATURES].fillna(0)
y = (df["failure_probability"] > TARGET_THRESHOLD).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"{len(df)} rows | train {len(X_train)} / test {len(X_test)} | positive rate {y.mean():.1%}")

## 3. Train and log to MLflow

In [ ]:
params = {"n_estimators": 100, "max_depth": 6, "random_state": 42}

with mlflow.start_run(run_name="random_forest_baseline"):
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Accuracy: {accuracy:.4f}\n")
    print(classification_report(y_test, predictions))

    mlflow.log_params(params)
    mlflow.log_param("target_threshold", TARGET_THRESHOLD)
    mlflow.log_metric("accuracy", accuracy)
    for feature, importance in zip(FEATURES, model.feature_importances_):
        mlflow.log_metric(f"importance_{feature}", importance)

    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="railops_model",
        signature=infer_signature(X_train, model.predict(X_train)),
        input_example=X_train.iloc[:5],
        registered_model_name=REGISTERED_MODEL_NAME,
    )
    print(f"Logged and registered '{REGISTERED_MODEL_NAME}' in MLflow.")

## 4. Save the model for Vayu Model Registry

In [ ]:
joblib.dump(model, MODEL_PATH)
print(f"Saved {MODEL_PATH} — continue to Step 5 (05_model_registry/upload_model.py).")